# HEINZO Inpainting

⚠️ This colab is specifically designed for generating safe and appropriate content, adhering strictly to guidelines for creating family-friendly material. It's not programmed to generate any content that could be considered inappropriate or explicit.

# 1. Install

In [ ]:
# @title Install Requirements
!pip install -q diffusers transformers accelerate controlnet_aux safetensors
!pip install -q opencv-python pillow numpy gradio
%cd /content
!mkdir -p /content/images

In [ ]:
# @title Load Diffusers
import torch
from diffusers import StableDiffusionControlNetInpaintPipeline, ControlNetModel, DPMSolverMultistepScheduler
from datetime import datetime

# 1. Load ControlNet
controlnet = ControlNetModel.from_pretrained(
    "lllyasviel/sd-controlnet-openpose", torch_dtype=torch.float16
)

# 2. Load Pipeline
model_id = "Uminosachi/realisticVisionV51_v51VAE-inpainting"
pipe = StableDiffusionControlNetInpaintPipeline.from_pretrained(
    model_id, controlnet=controlnet, torch_dtype=torch.float16
).to("cuda")

# 3. Set Scheduler for Realism
pipe.scheduler = DPMSolverMultistepScheduler.from_config(pipe.scheduler.config, use_karras_sigmas=True)
pipe.safety_checker = None

# 2a. Masking

In [ ]:
# @title Gradio
import gradio as gr
import numpy as np
from PIL import Image

def process_mask(data):
    global full_res_input, full_res_mask, input_image, mask_image

    full_res_input = data['background'].convert("RGB")

    # Calculate legal dimensions here
    ai_w, ai_h = get_scaled_dimensions(*full_res_input.size)

    if len(data['layers']) > 0:
        mask_layer = data['layers'][0].convert("RGBA")
        mask_alpha = mask_layer.split()[-1]
        full_res_mask = mask_alpha.point(lambda p: 255 if p > 0 else 0)
    else:
        full_res_mask = Image.new("L", full_res_input.size, 0)

    # Use the pre-calculated ai_w and ai_h
    input_image = full_res_input.resize((ai_w, ai_h), Image.LANCZOS)
    mask_image = full_res_mask.resize((ai_w, ai_h), Image.NEAREST)

    return f"Processed! AI resolution set to {ai_w}x{ai_h}."

# Create the Gradio Interface
with gr.Blocks() as demo:
    gr.Markdown("### Upload and Paint your Mask\nUse the brush tool to cover the area you want to change.")

    # ImageEditor is the modern Gradio way to handle masking
    img_input = gr.ImageEditor(
        type="pil",
        label="Input Image",
        sources=["upload"],
        brush=gr.Brush(colors=["#FFFFFF"], color_mode="fixed"), # We only need white for masks
        eraser=True
    )

    status = gr.Textbox(label="Status", interactive=False)
    btn = gr.Button("Send to Pipeline", variant="primary")

    btn.click(process_mask, inputs=[img_input], outputs=[status])

demo.launch(inline=True, debug=False, height=600)

# 2b. (alternative)Manual Upload

In [ ]:
# @title Upload Input Image
from google.colab import files
from PIL import Image
import io

print("Upload your source image:")
uploaded_input = files.upload()
input_image_path = list(uploaded_input.keys())[0]

def get_scaled_dimensions(width, height, max_size=768):
    # 1. Calculate the scaling factor
    ratio = min(max_size / width, max_size / height)

    # 2. Calculate new dimensions
    new_w = int(width * ratio)
    new_h = int(height * ratio)

    # 3. Snap to nearest multiple of 8 (floor division)
    new_w = (new_w // 8) * 8
    new_h = (new_h // 8) * 8

    return new_w, new_h

# --- Apply to your Step 2 or 3b ---
full_res_input = Image.open(io.BytesIO(uploaded_input[input_image_path])).convert("RGB")

# Get legal AI dimensions
ai_w, ai_h = get_scaled_dimensions(*full_res_input.size)

# Create the AI version
input_image = full_res_input.resize((ai_w, ai_h), Image.LANCZOS)

print(f"Full Res: {full_res_input.size}")
print(f"AI Res (Legal): {input_image.size}") # This will now be 768x608 or similar
display(input_image)

In [ ]:
# @title Upload Mask Image
print("Upload your mask image:")
uploaded_mask = files.upload()
mask_image_path = list(uploaded_mask.keys())[0]

# Keep full res mask
full_res_mask = Image.open(io.BytesIO(uploaded_mask[mask_image_path])).convert("L")
full_res_mask = full_res_mask.resize(full_res_input.size) # Match input exactly

# Smaller version for AI
mask_image = full_res_mask.resize(input_image.size, Image.NEAREST)
print(f"Mask Image: {mask_image.size}")
display(mask_image)

# 3. ControlNet & Blur

In [ ]:
# @title ControlNet
from controlnet_aux import OpenposeDetector

openpose = OpenposeDetector.from_pretrained("lllyasviel/ControlNet")
pose_image = openpose(input_image)

print("Detected Pose:")
display(pose_image)

In [ ]:
# @title Set Blur
from PIL import ImageFilter

mask_blur_radius = 15 # @param {type:"slider", min:0, max:100, step:1}
mask_image = mask_image.filter(ImageFilter.GaussianBlur(mask_blur_radius))

print("Blurred Mask:")
display(mask_image)

# 4. Generate

In [ ]:
# @title Set Parameters


# --- Parameters ---
prompt = "(8k, RAW photo, best quality, masterpiece:1.2), (realistic, photo-realistic:1.37), 1girl, medium breast, titfuck, paizuri, professional lighting, radiosity, physically-based rendering, " # @param {type:"string"}
# @markdown ---
negative_prompt = "necklace, clothes, clothing, paintings, sketches, (worst quality:2), (low quality:2), (normal quality:2), lowres, normal quality, ((monochrome)), ((grayscale)), ( clenched teeth,  ), skin spots, acnes, skin blemishes, age spot, bad anatomy, ugly face, deformed features, derp face, bad eyes, " # @param {type:"string"}
# @markdown ---
steps = 25 # @param {type:"slider", min:1, max:100, step:1}
guidance = 7 # @param {type:"slider", min:1, max:25, step:0.5}
denoising_strength = 1 # @param {type:"slider", min:0.1, max:1, step:0.05}

num_images = 2 # @param {type:"number"}
# Load LoRA
lora_path = "nemesis1/titfuck" # @param ["nemesis1/oversizedshirt", "nemesis1/betterbody", "nemesis1/hourglassbody", "nemesis1/breastinclassbetter_v141", "nemesis1/cumshot", "nemesis1/cohf", "nemesis1/skirtlift", "nemesis1/ShirtLift", "nemesis1/calvinklein", "nemesis1/sm", "nemesis1/croptop_underboob", "nemesis1/bikini_micro", "nemesis1/bikini_1", "nemesis1/bikini_underboob", "nemesis1/bikini_furcoat", "nemesis1/onoff_v4", "nemesis1/nipples"] {allow-input: true}
pipe.load_lora_weights(lora_path)
lora_weight = 1 # @param {type:"slider", min:0, max:1, step:0.1}

# @markdown ---
width, height = input_image.size

In [ ]:
# @title Generate
import numpy as np
import cv2
from PIL import ImageFilter
tgl = datetime.now().strftime("%y%m%d_%H%M%S")
# 1. Run the AI Inference (same as before)
images = pipe(
    prompt=prompt,
    negative_prompt=negative_prompt,
    image=input_image,
    control_image=pose_image,
    mask_image=mask_image,
    cross_attention_kwargs={"scale": lora_weight},
    height=input_image.height,
    width=input_image.width,
    num_inference_steps=steps,
    guidance_scale=guidance,
    num_images_per_prompt=num_images,
    strength=denoising_strength,
).images

# 2. Patching Logic
for i in range(num_images):
    # 1. Upscale the AI generation back to the original photo's dimensions
    generated_upscaled = images[i].resize(full_res_input.size, Image.LANCZOS)

    # Prepare arrays
    original_array = np.array(full_res_input)
    generated_array = np.array(generated_upscaled)

    # --- FIX START ---
    # 2. Resize the mask to match the high-res photo dimensions first!
    resized_mask = full_res_mask.resize(full_res_input.size, Image.LANCZOS)

    # Now apply the blur to the correctly sized mask
    mask_for_blend = resized_mask.filter(ImageFilter.GaussianBlur(radius=4))
    # --- FIX END ---

    # 3. Convert mask to float alpha
    mask_alpha = np.array(mask_for_blend).astype(float) / 255.0

    # Ensure mask_alpha has 3 channels to match (H, W, 3)
    if len(mask_alpha.shape) == 2:
        mask_alpha = np.stack([mask_alpha] * 3, axis=-1)

    # 4. Composite: keep original pixels where mask is black, use AI pixels where mask is white
    final_array = (original_array * (1 - mask_alpha) + generated_array * mask_alpha).astype(np.uint8)

    final_image = Image.fromarray(final_array)

    print(f"Final Output Resolution: {final_image.size}")
    display(final_image)
    final_image.save(f"/content/images/H_{tgl}_{i}.png")

# Download

In [ ]:
# @title ⬇️ 7. Download all images as rar (cukup sekali diakhir)
!zip -r /content/H_Inpainting.zip /content/images
from google.colab import files
files.download("/content/H_Inpainting.zip")